# Donor Upgrade Potential Pipeline (Model 2) — CRISP-DM

Follows **CRISP-DM** (Business Understanding through Deployment), consistent with the course ML pipeline template.

## Phase 1: Business Understanding
**Objective:** Predict whether the **next monetary gift** exceeds the donor's historical median (upgrade).
**Stakeholders:** Fundraising / admin.
**Success metrics:** ROC-AUC, AP, F1 vs **DummyClassifier** baseline on a chronological holdout.

## Phase 2: Data Understanding
**Sources:** `donations.csv`, `supporters.csv` (`ml-pipelines/data/README.md`). EDA after load in code.

## Phase 3: Data Preparation
Historical-only features; label from next gift only; chronological split; sklearn `Pipeline` preprocessing fit on train.

## Phase 4: Modeling
Random Forest + Logistic Regression.

## Phase 5: Evaluation
Metrics + majority-class baseline; permutation importance.

## Phase 6: Deployment
**joblib** artifacts below. Production: `donor_models_train.py`, `/api/ml/donor/train`, `DonorUpgradeInsightsPage`.


### Row construction (leakage-safe)

**Goal:** Predict if the donor's next **monetary** gift exceeds their **historical median** at snapshot.

- Baseline (median) from historical gifts only.
- Next gift used **only** for the label.
- Chronological split by `snapshot_date`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
import joblib


def resolve_csv(name: str) -> Path:
    cwd = Path.cwd()
    for base in (cwd, cwd / "ml-pipelines", cwd / "data", cwd / "ml-pipelines" / "data"):
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find {name}. Place it in ., data/, ml-pipelines/, or ml-pipelines/data/ (see ml-pipelines/data/README.md)."
    )


DATA_DIR = resolve_csv("donations.csv").parent
ARTIFACTS_DIR = (Path("ml-pipelines") / "artifacts") if (Path.cwd() / "ml-pipelines").is_dir() else (DATA_DIR / "artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

donations = pd.read_csv(DATA_DIR / "donations.csv", parse_dates=["donation_date"])
supporters = pd.read_csv(DATA_DIR / "supporters.csv", parse_dates=["created_at", "first_donation_date"])

print("=== Phase 2: EDA ===")
print("donations:", donations.shape, "| monetary rows:", len(donations[donations["donation_type"] == "Monetary"]))

def time_split(df, time_col, frac=0.8):
    df = df.sort_values(time_col).copy()
    cut = int(len(df) * frac)
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

def build_preprocessor(X):
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('pow', PowerTransformer(method='yeo-johnson', standardize=False)), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])


In [ ]:
m = donations[donations['donation_type'] == 'Monetary'].sort_values(['supporter_id', 'donation_date']).copy()
rows = []
for sid, g in m.groupby('supporter_id'):
    g = g.reset_index(drop=True)
    if len(g) < 3:
        continue
    for i in range(1, len(g)-1):
        snapshot_date = g.loc[i, 'donation_date']
        hist = g.loc[:i]            # historical-only features
        next_amt = g.loc[i+1, 'amount']  # only label source
        baseline = hist['amount'].median()
        rows.append({
            'supporter_id': sid,
            'snapshot_date': snapshot_date,
            'upgrade_next_gift': int(next_amt > baseline),
            'hist_median_amount': baseline,
            'hist_mean_amount': hist['amount'].mean(),
            'hist_std_amount': hist['amount'].std(),
            'gift_count_hist': len(hist),
            'recurring_rate_hist': hist['is_recurring'].astype(int).mean(),
            'social_channel_share_hist': (hist['channel_source'] == 'SocialMedia').mean(),
        })

m2 = pd.DataFrame(rows).merge(
    supporters[['supporter_id', 'supporter_type', 'relationship_type', 'region', 'country', 'acquisition_channel']],
    on='supporter_id', how='left'
)

train_df, test_df = time_split(m2, 'snapshot_date', 0.8)
X_train = train_df.drop(columns=['upgrade_next_gift', 'snapshot_date'])
y_train = train_df['upgrade_next_gift']
X_test = test_df.drop(columns=['upgrade_next_gift', 'snapshot_date'])
y_test = test_df['upgrade_next_gift']

pre = build_preprocessor(X_train)
predictive = Pipeline([('pre', pre), ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=4, random_state=42))])
explanatory = Pipeline([('pre', pre), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])

predictive.fit(X_train, y_train)
explanatory.fit(X_train, y_train)

prob = predictive.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
db_pred = dummy.predict(X_test)
db_proba = dummy.predict_proba(X_test)[:, 1]
print("=== Phase 5: Baseline DummyClassifier (test) ===")
print({"f1": f1_score(y_test, db_pred), "roc_auc": roc_auc_score(y_test, db_proba) if y_test.nunique() > 1 else None})

metrics = {
    'avg_precision': average_precision_score(y_test, prob),
    'f1': f1_score(y_test, pred),
    'test_positive_rate': float(y_test.mean()),
    'test_classes': sorted(y_test.unique().tolist()),
}
if y_test.nunique() > 1:
    metrics['roc_auc'] = roc_auc_score(y_test, prob)
else:
    metrics['roc_auc'] = None
    print('Warning: ROC-AUC skipped because test set has only one class.')

print("=== Phase 5: Random Forest (test) ===")
print(metrics)

imp = permutation_importance(predictive, X_test, y_test, n_repeats=8, random_state=42)
print(pd.DataFrame({'feature': X_test.columns, 'importance': imp.importances_mean}).sort_values('importance', ascending=False).head(10))

joblib.dump(predictive, ARTIFACTS_DIR / 'model2_predictive.joblib')
joblib.dump(explanatory, ARTIFACTS_DIR / 'model2_explanatory.joblib')


In [ ]:
# Final business insights block (human-readable + actionable)

print('\n=== BUSINESS TAKEAWAYS: MODEL 2 (DONOR UPGRADE POTENTIAL) ===')

if 'predictive' not in globals():
    print('Model was not trained successfully. Run the previous cell and check warnings/errors first.')
else:
    scored = m2.copy()
    features = [c for c in scored.columns if c not in ['upgrade_next_gift', 'snapshot_date']]
    scored['upgrade_score'] = predictive.predict_proba(scored[features])[:, 1]

    scored['upgrade_band'] = pd.cut(scored['upgrade_score'], bins=[-0.001, 0.35, 0.65, 1.0], labels=['Low', 'Medium', 'High'])
    print('Upgrade band distribution:')
    print(scored['upgrade_band'].value_counts(dropna=False).rename_axis('upgrade_band').reset_index(name='count').to_string(index=False))

    # Top upgrade candidates
    cands = scored.sort_values('upgrade_score', ascending=False).head(20).copy()
    cands['suggested_ask_floor'] = cands['hist_median_amount'] * 1.10
    cands['suggested_ask_ceiling'] = cands['hist_median_amount'] * 1.30
    print('\nTop 20 upgrade candidates (ask-ladder worklist):')
    print(cands[['supporter_id', 'upgrade_score', 'supporter_type', 'acquisition_channel', 'hist_median_amount', 'suggested_ask_floor', 'suggested_ask_ceiling', 'gift_count_hist']].to_string(index=False))

    # Segment opportunity
    seg = scored.groupby(['supporter_type', 'acquisition_channel'], dropna=False)['upgrade_score'].mean().reset_index().sort_values('upgrade_score', ascending=False)
    print('\nHighest-upgrade-opportunity segments:')
    print(seg.head(10).to_string(index=False))

    print('\nActionable guidance:')
    print('- Ask top candidates with a structured 10-30% ask ladder above their historical median.')
    print('- Use segment table to prioritize campaign design and message framing.')
    print('- Start with donors who are both high upgrade score and have stable giving history.')
    print('- Treat this as a prioritization model, not a guarantee of gift increase.')